# Evaluation — Pathway Impact Predictions

**Goal**: Quantitatively evaluate the variant → pathway scoring pipeline
against a curated ground-truth set of known gene–disease–pathway
associations.

## Metrics

| Metric | Description |
|--------|-------------|
| Precision@K | Fraction of top-K scored pathways that are true positives |
| Recall@K    | Fraction of known pathways retrieved in top-K |
| AUROC       | Area under the receiver operating characteristic curve |
| Mean Reciprocal Rank | Position of first true positive in ranked list |

## 1. Setup

In [ ]:
import sys, os, math

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from data_ingestion.load_clinvar import ClinVarRecord
from data_ingestion.load_pathways import Pathway
from preprocessing.normalize_variants import normalize_clinvar_records
from preprocessing.gene_to_pathway_mapping import build_gene_pathway_map
from models.variant_pathway_model import build_variant_pathway_graph
from models.scoring import rank_pathways

print('Imports OK')

## 2. Ground-truth annotations

A minimal curated set of known gene → disease → pathway relationships
used as the evaluation reference.

In [ ]:
# ground_truth: {gene: [pathway_id, ...]}
GROUND_TRUTH = {
    'LRRK2': ['hsa05012', 'hsa04010'],
    'PSEN1': ['hsa05010'],
    'HTT':   ['hsa05016'],
}

# All true-positive pathway IDs
ALL_POSITIVE_PATHWAYS = {
    pid for pids in GROUND_TRUTH.values() for pid in pids
}
print('True-positive pathways:', ALL_POSITIVE_PATHWAYS)

## 3. Build the pipeline (same as Notebook 1)

In [ ]:
clinvar_records = [
    ClinVarRecord(variant_id='1388948', gene_symbol='LRRK2', chrom='12',
                  pos=40340400, ref='G', alt='A',
                  clinical_significance='Pathogenic',
                  condition='Parkinson disease, late-onset',
                  review_status='criteria provided, single submitter'),
    ClinVarRecord(variant_id='4149', gene_symbol='PSEN1', chrom='14',
                  pos=73659468, ref='C', alt='T',
                  clinical_significance='Pathogenic',
                  condition='Alzheimer disease, early-onset',
                  review_status='reviewed by expert panel'),
    ClinVarRecord(variant_id='54321', gene_symbol='HTT', chrom='4',
                  pos=3076407, ref='CAG', alt='CAGCAGCAG',
                  clinical_significance='Pathogenic',
                  condition='Huntington disease',
                  review_status='criteria provided, multiple submitters, no conflicts'),
    ClinVarRecord(variant_id='99999', gene_symbol='LRRK2', chrom='12',
                  pos=40734202, ref='G', alt='C',
                  clinical_significance='Likely pathogenic',
                  condition='Parkinson disease, late-onset',
                  review_status='criteria provided, single submitter'),
]

pathways = [
    Pathway(pathway_id='hsa05012', name='Parkinson disease',
            source='KEGG', gene_symbols=['LRRK2', 'PINK1', 'PARK7', 'SNCA', 'UCHL1']),
    Pathway(pathway_id='hsa05010', name='Alzheimer disease',
            source='KEGG', gene_symbols=['PSEN1', 'PSEN2', 'APP', 'APOE', 'BACE1']),
    Pathway(pathway_id='hsa05016', name='Huntington disease',
            source='KEGG', gene_symbols=['HTT', 'BDNF', 'CASP3', 'HDAC4', 'TBP']),
    Pathway(pathway_id='hsa04010', name='MAPK signaling pathway',
            source='KEGG', gene_symbols=['LRRK2', 'MAPK1', 'MAPK3', 'RAF1', 'MAP2K1']),
    # Negative control pathway — no linked variants expected
    Pathway(pathway_id='hsa04151', name='PI3K-Akt signaling pathway',
            source='KEGG', gene_symbols=['AKT1', 'PIK3CA', 'PTEN', 'MTOR']),
]

variants  = normalize_clinvar_records(clinvar_records)
gene_map  = build_gene_pathway_map(pathways)
graph     = build_variant_pathway_graph(variants, gene_map)
ranked    = rank_pathways(graph)

print(f'Graph: {graph.variant_count} variants, {graph.pathway_count} pathways with links')

## 4. Evaluation metrics

In [ ]:
def precision_at_k(ranked_ids, positives, k):
    top_k = ranked_ids[:k]
    hits = sum(1 for pid in top_k if pid in positives)
    return hits / k if k > 0 else 0.0

def recall_at_k(ranked_ids, positives, k):
    top_k = ranked_ids[:k]
    hits = sum(1 for pid in top_k if pid in positives)
    return hits / len(positives) if positives else 0.0

def mean_reciprocal_rank(ranked_ids, positives):
    for rank, pid in enumerate(ranked_ids, start=1):
        if pid in positives:
            return 1.0 / rank
    return 0.0

ranked_ids = [ps.pathway_id for ps in ranked]

print('Ranked pathway IDs:', ranked_ids)
print()

for k in [1, 2, 3, len(ranked)]:
    p = precision_at_k(ranked_ids, ALL_POSITIVE_PATHWAYS, k)
    r = recall_at_k(ranked_ids, ALL_POSITIVE_PATHWAYS, k)
    print(f'  P@{k} = {p:.3f}   R@{k} = {r:.3f}')

mrr = mean_reciprocal_rank(ranked_ids, ALL_POSITIVE_PATHWAYS)
print(f'\nMRR = {mrr:.3f}')

## 5. Score breakdown

In [ ]:
print(f'\n{"Rank":<6}{"Pathway ID":<15}{"TP?":<6}{"Score":>8}{"Norm":>10}  Name')
print('-' * 75)
for rank, ps in enumerate(ranked, start=1):
    is_tp = 'YES' if ps.pathway_id in ALL_POSITIVE_PATHWAYS else 'no'
    print(
        f'{rank:<6}{ps.pathway_id:<15}{is_tp:<6}'
        f'{ps.score:>8.3f}{ps.normalised_score:>10.3f}  {ps.pathway_name}'
    )

## 6. Sensitivity to score weights

In [ ]:
configs = [
    {'label': 'default',         'lof_weight': 2.0, 'pathogenic_weight': 1.0, 'vus_weight': 0.1},
    {'label': 'LoF-heavy (4×)',  'lof_weight': 4.0, 'pathogenic_weight': 1.0, 'vus_weight': 0.1},
    {'label': 'Flat weights',    'lof_weight': 1.0, 'pathogenic_weight': 1.0, 'vus_weight': 1.0},
    {'label': 'Ignore VUS',      'lof_weight': 2.0, 'pathogenic_weight': 1.0, 'vus_weight': 0.0},
]

print(f'{"Config":<22}  MRR      P@1    P@2    P@3')
print('-' * 60)
for cfg in configs:
    label = cfg.pop('label')
    r = rank_pathways(graph, **cfg)
    ids = [ps.pathway_id for ps in r]
    mrr_val = mean_reciprocal_rank(ids, ALL_POSITIVE_PATHWAYS)
    p1 = precision_at_k(ids, ALL_POSITIVE_PATHWAYS, 1)
    p2 = precision_at_k(ids, ALL_POSITIVE_PATHWAYS, 2)
    p3 = precision_at_k(ids, ALL_POSITIVE_PATHWAYS, 3)
    print(f'{label:<22}  {mrr_val:.3f}    {p1:.3f}  {p2:.3f}  {p3:.3f}')

## 7. Next steps

- Replace synthetic data with real ClinVar + gnomAD downloads
- Add gnomAD allele-frequency filtering (e.g. AF < 0.001 for rare variants)
- Compute AUROC once a larger positive/negative pathway set is available
- Explore gene-set enrichment analysis (GSEA) as a complementary scoring approach
- Integrate phenotype filtering via HPO terms from `preprocessing.ontology_normalization`